# Stage 3A: Merge Features into Model-Ready Arrays

Combines PCA-reduced dynamic features (from Phase 6.3) with static OSM/VIIRS features into the numpy arrays that the hybrid model and SHAP script expect.

**Key difference from training Phase 4-2:** The dynamic features have already been PCA-reduced (4096 to 256) during feature extraction in Phase 6.3. So no PCA transform is needed here. The saved `final_pca.pkl` was already applied.

**Inputs:**
- `dynamic_features_2025_prediction_points.csv` (PointID, Quarter, CNN_0..CNN_255) -- from Phase 6.3
- `osm_features_2025.csv` (PointID, 19 OSM features + VIIRS_Median) -- from Stage 2
- `prediction_points.csv` (PointID, Lat, Lon, Locality, Province, etc.) -- from Stage 1
- `static_feature_names.txt` (from training) -- to enforce column order

**Outputs (uploaded to GCS for the SHAP script):**
- `X_dynamic.npy` -- shape (N, 4, 256), already PCA-reduced
- `X_static.npy` -- shape (N, 20), raw (scaler applied at inference time)
- `cluster_ids.npy` -- PointIDs in the same row order
- `static_feature_names.txt` -- copied from training

In [1]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================
# 1. PATHS
# ============================================================

# Dynamic features (PCA-reduced, from Phase 6.3 on Colab)
DYNAMIC_CSV = '/Users/ruben/Desktop/Thesis/2025Data/dynamic_features_2025_prediction_points.csv'

# Static features (OSM + VIIRS, from Stage 2)
STATIC_CSV = '/Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_expanded.csv'

# Prediction points metadata
POINTS_CSV = '/Users/ruben/Desktop/Thesis/2025Data/prediction_points.csv'

# Static feature names from training (defines column order)
NAMES_FILE = '/Users/ruben/Desktop/Thesis/2025Data/static_feature_names.txt'

# Output directory
OUTPUT_DIR = '/Users/ruben/Desktop/Thesis/2025Data/merged_expanded'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# PCA dimensionality (from Phase 6.3 output)
N_COMPONENTS = 256

print("Paths configured.")

Paths configured.


## 2. Load Data

In [3]:
# ============================================================
# 2A. LOAD DYNAMIC FEATURES
# ============================================================

print("Loading dynamic features (PCA-reduced)...")
df_dyn = pd.read_csv(DYNAMIC_CSV)
print(f"  Records: {len(df_dyn)}")
print(f"  Columns: {df_dyn.shape[1]}")
print(f"  Unique PointIDs: {df_dyn['PointID'].nunique()}")
print(f"  Quarters: {sorted(df_dyn['Quarter'].unique())}")

# Verify feature count matches expected PCA dimensions
cnn_cols = [c for c in df_dyn.columns if c.startswith('CNN_')]
print(f"  CNN feature columns: {len(cnn_cols)} (expected {N_COMPONENTS})")
assert len(cnn_cols) == N_COMPONENTS, (
    f"Expected {N_COMPONENTS} CNN columns, got {len(cnn_cols)}. "
    f"Check N_COMPONENTS setting."
)

Loading dynamic features (PCA-reduced)...
  Records: 4448
  Columns: 258
  Unique PointIDs: 1112
  Quarters: ['Q1', 'Q2', 'Q3', 'Q4']
  CNN feature columns: 256 (expected 256)


In [4]:
# ============================================================
# 2B. LOAD STATIC FEATURES
# ============================================================

print("Loading static features (OSM + VIIRS)...")
df_stat = pd.read_csv(STATIC_CSV)
print(f"  Records: {len(df_stat)}")
print(f"  Columns: {df_stat.columns.tolist()}")

# Load expected column order from training
with open(NAMES_FILE) as f:
    training_feature_names = [line.strip() for line in f]
print(f"\nTraining feature order ({len(training_feature_names)} features):")
for i, name in enumerate(training_feature_names):
    present = 'OK' if name in df_stat.columns else 'MISSING'
    print(f"  {i+1:2d}. {name:30s} [{present}]")

Loading static features (OSM + VIIRS)...
  Records: 1240
  Columns: ['PointID', 'Total_Road_Length', 'Main_Roads_Length', 'Secondary_Roads_Length', 'Local_Roads_Length', 'Tracks_Length', 'Total_Bldg_Count', 'Total_Bldg_Area', 'Mean_Bldg_Area', 'Total_Bldg_Proportion', 'Bldg_Density_per_km2', 'Bldg_residential_Count', 'Bldg_residential_TotalArea', 'Bldg_residential_MeanArea', 'Bldg_residential_Proportion', 'Bldg_commercial_Count', 'Bldg_commercial_TotalArea', 'Bldg_commercial_MeanArea', 'Bldg_commercial_Proportion', 'Bldg_industrial_Count', 'Bldg_industrial_TotalArea', 'Bldg_industrial_MeanArea', 'Bldg_industrial_Proportion', 'Bldg_school_Count', 'Bldg_school_TotalArea', 'Bldg_school_MeanArea', 'Bldg_school_Proportion', 'Bldg_hospital_Count', 'Bldg_hospital_TotalArea', 'Bldg_hospital_MeanArea', 'Bldg_hospital_Proportion', 'Total_POI_Count', 'POI_bank_Count', 'POI_atm_Count', 'POI_hotel_Count', 'POI_fast_food_Count', 'POI_convenience_Count', 'POI_restaurant_Count', 'POI_market_Count', 'P

In [5]:
# ============================================================
# 2C. LOAD PREDICTION POINTS METADATA
# ============================================================

print("Loading prediction points...")
df_points = pd.read_csv(POINTS_CSV)
print(f"  Total points: {len(df_points)}")
print(f"  Provinces: {df_points['Province'].nunique()}")

Loading prediction points...
  Total points: 1240
  Provinces: 11


## 3. Identify Valid Points

In [6]:
# ============================================================
# 3. FIND POINTS WITH COMPLETE DATA IN BOTH SETS
# ============================================================

# Points with all 4 quarters in dynamic features
quarter_counts = df_dyn.groupby('PointID').size()
complete_dynamic = set(
    quarter_counts[quarter_counts == 4].index
)

# Identify the ID column in static CSV
# It might be 'PointID' or 'DHSCLUST' depending on how you ran OSM extraction
if 'PointID' in df_stat.columns:
    stat_id_col = 'PointID'
elif 'DHSCLUST' in df_stat.columns:
    stat_id_col = 'DHSCLUST'
else:
    raise ValueError(
        f"Cannot find ID column in static CSV. "
        f"Columns: {df_stat.columns.tolist()}"
    )
print(f"Static ID column: '{stat_id_col}'")

available_static = set(df_stat[stat_id_col].unique())

# Intersection
valid_ids = sorted(complete_dynamic & available_static)

print(f"\nComplete dynamic (4 quarters): {len(complete_dynamic)}")
print(f"Available in static:           {len(available_static)}")
print(f"Valid (both):                  {len(valid_ids)}")

# Report what was dropped
dropped_dyn = complete_dynamic - available_static
dropped_stat = available_static - complete_dynamic
if dropped_dyn:
    print(f"\nDropped (in dynamic, not in static): {len(dropped_dyn)}")
if dropped_stat:
    print(f"Dropped (in static, not in dynamic): {len(dropped_stat)}")

Static ID column: 'PointID'

Complete dynamic (4 quarters): 1112
Available in static:           1240
Valid (both):                  1112
Dropped (in static, not in dynamic): 128


## 4. Reshape Dynamic Features

In [7]:
# ============================================================
# 4. RESHAPE DYNAMIC INTO (N, 4, 256)
# ============================================================
# These are ALREADY PCA-reduced from Phase 6.3.
# No further PCA transform needed.

print("Reshaping dynamic features...")

X_dynamic = []
valid_clusters_ordered = []
skipped = []

for pid in valid_ids:
    subset = df_dyn[
        df_dyn['PointID'] == pid
    ].sort_values('Quarter')

    if len(subset) != 4:
        skipped.append((pid, len(subset)))
        continue

    feats = subset[cnn_cols].values  # shape: (4, 256)
    X_dynamic.append(feats)
    valid_clusters_ordered.append(pid)

X_dynamic = np.array(X_dynamic, dtype=np.float32)

print(f"X_dynamic shape: {X_dynamic.shape}")
print(f"  Expected: ({len(valid_ids)}, 4, {N_COMPONENTS})")
print(f"  Skipped: {len(skipped)}")
if skipped:
    print(f"  Skipped IDs: {skipped[:10]}")

Reshaping dynamic features...
X_dynamic shape: (1112, 4, 256)
  Expected: (1112, 4, 256)
  Skipped: 0


## 5. Align Static Features

In [8]:
# ============================================================
# 5. ALIGN STATIC FEATURES IN TRAINING COLUMN ORDER
# ============================================================
# Critical: columns must be in the exact same order as training.
# The scaler and model weights depend on this ordering.

print("Aligning static features...")

# Log transform skewed features in static
log_cols = [
    'LU_Residential_m2', 'LU_Commercial_m2',
    'LU_Industrial_m2',  'LU_Agricultural_m2',
    'LU_Forest_m2',
    'POI_restaurant_Count', 'Total_POI_Count',
]
for col in log_cols:
    df_stat[col] = np.log1p(df_stat[col])

# Filter to valid IDs and set index
df_stat_aligned = (
    df_stat[df_stat[stat_id_col].isin(valid_clusters_ordered)]
    .set_index(stat_id_col)
    .reindex(valid_clusters_ordered)
)

# Enforce training column order
missing_cols = [
    c for c in training_feature_names
    if c not in df_stat_aligned.columns
]
extra_cols = [
    c for c in df_stat_aligned.columns
    if c not in training_feature_names
]

if missing_cols:
    print(f"  WARNING: Missing columns (will be zero-filled): {missing_cols}")
    for col in missing_cols:
        df_stat_aligned[col] = 0.0

if extra_cols:
    print(f"  Extra columns (will be dropped): {extra_cols}")

# Reorder to match training exactly
df_stat_aligned = df_stat_aligned[training_feature_names]

X_static = df_stat_aligned.values.astype(np.float32)

print(f"X_static shape: {X_static.shape}")
print(f"  Expected: ({len(valid_clusters_ordered)}, {len(training_feature_names)})")
print(f"  Column order: {list(df_stat_aligned.columns)[:5]} ...")

Aligning static features...
X_static shape: (1112, 52)
  Expected: (1112, 52)
  Column order: ['Total_Road_Length', 'Main_Roads_Length', 'Secondary_Roads_Length', 'Local_Roads_Length', 'Tracks_Length'] ...


## 6. Validate

In [9]:
# ============================================================
# 6. VALIDATE MERGED DATASET
# ============================================================

cluster_ids = np.array(valid_clusters_ordered)

print("Validating merged dataset...")

# Check shapes
assert X_dynamic.shape[0] == X_static.shape[0] == len(cluster_ids), (
    f"Row mismatch: dynamic={X_dynamic.shape[0]}, "
    f"static={X_static.shape[0]}, ids={len(cluster_ids)}"
)
print(f"  Row alignment: OK ({len(cluster_ids)} points)")

# Check for NaN in dynamic
nan_dyn = np.isnan(X_dynamic).sum()
if nan_dyn > 0:
    print(f"  WARNING: {nan_dyn} NaN in dynamic features. Filling with 0.")
    X_dynamic = np.nan_to_num(X_dynamic, nan=0.0)
else:
    print(f"  Dynamic NaN: none")

# Check for NaN in static
nan_stat = np.isnan(X_static).sum()
if nan_stat > 0:
    print(f"  WARNING: {nan_stat} NaN in static features. Filling with column means.")
    col_means = np.nanmean(X_static, axis=0)
    for j in range(X_static.shape[1]):
        mask = np.isnan(X_static[:, j])
        X_static[mask, j] = col_means[j]
else:
    print(f"  Static NaN: none")

# Summary stats
print(f"\nFinal dataset:")
print(f"  X_dynamic : {X_dynamic.shape} "
      f"(already PCA-reduced to {N_COMPONENTS} dims)")
print(f"  X_static  : {X_static.shape}")
print(f"  cluster_ids: {cluster_ids.shape}")
print(f"  Dynamic value range: "
      f"[{X_dynamic.min():.2f}, {X_dynamic.max():.2f}]")
print(f"  Static value range:  "
      f"[{X_static.min():.2f}, {X_static.max():.2f}]")

# Per-province counts
df_pts = df_points[df_points['PointID'].isin(cluster_ids)]
print(f"\nPoints per province:")
for prov, count in df_pts['Province'].value_counts().items():
    print(f"  {prov}: {count}")

Validating merged dataset...
  Row alignment: OK (1112 points)
  Dynamic NaN: none
  Static NaN: none

Final dataset:
  X_dynamic : (1112, 4, 256) (already PCA-reduced to 256 dims)
  X_static  : (1112, 52)
  cluster_ids: (1112,)
  Dynamic value range: [-49.17, 72.95]
  Static value range:  [0.00, 11771847.00]

Points per province:
  Zamboanga del Norte: 187
  Davao Oriental: 157
  NCR: 149
  Ilocos Norte: 119
  Kalinga: 110
  Benguet: 101
  Pampanga: 84
  Maguindanao del Sur: 83
  Aklan: 51
  Basilan: 42
  Tawi-Tawi: 29


## 7. Save

In [10]:
# ============================================================
# 7. SAVE NUMPY ARRAYS
# ============================================================

np.save(f'{OUTPUT_DIR}/X_dynamic.npy', X_dynamic)
np.save(f'{OUTPUT_DIR}/X_static.npy', X_static)
np.save(f'{OUTPUT_DIR}/cluster_ids.npy', cluster_ids)

# Copy the training feature names file
import shutil
shutil.copy(
    NAMES_FILE,
    f'{OUTPUT_DIR}/static_feature_names.txt'
)

print(f"Saved to: {OUTPUT_DIR}/")
print(f"  X_dynamic.npy          ({X_dynamic.nbytes / 1e6:.1f} MB)")
print(f"  X_static.npy           ({X_static.nbytes / 1e6:.1f} MB)")
print(f"  cluster_ids.npy        ({cluster_ids.nbytes / 1e3:.1f} KB)")
print(f"  static_feature_names.txt")

Saved to: /Users/ruben/Desktop/Thesis/2025Data/merged_expanded/
  X_dynamic.npy          (4.6 MB)
  X_static.npy           (0.2 MB)
  cluster_ids.npy        (8.9 KB)
  static_feature_names.txt


## 8. Important Note for SHAP Script

The `run_shap.py` cloud script currently applies PCA to `X_dynamic` at runtime:

```python
X_dynamic_pca = pca.transform(
    X_dynamic.reshape(-1, 4096)
).reshape(-1, 4, N_COMPONENTS)
```

**Since your dynamic features are already PCA-reduced (256 dims, not 4096), you must skip that PCA step.** Replace those lines in `run_shap.py` with:

```python
# Dynamic features are already PCA-reduced from Phase 6.3
N_COMPONENTS = X_dynamic.shape[2]  # 256
X_dynamic_pca = X_dynamic  # no transform needed
```

The scaler for static features still needs to be applied as before.

In [12]:
# ============================================================
# 9. UPLOAD TO GCS (optional, run if using cloud VM for SHAP)
# ============================================================
# Uncomment and run if you want to push directly to GCS.
# Otherwise, manually upload the merged/ folder.

import subprocess
GCS_DEST = 'gs://tala-sentinel2-data/data/merged'
for fname in ['X_dynamic.npy', 'X_static.npy',
               'cluster_ids.npy', 'static_feature_names.txt']:
     subprocess.run([
         'gsutil', 'cp',
         f'{OUTPUT_DIR}/{fname}',
         f'{GCS_DEST}/{fname}'
     ], check=True)
     print(f"  Uploaded: {fname}")
print(f"All files at: {GCS_DEST}")

Copying file:///Users/ruben/Desktop/Thesis/2025Data/merged/X_dynamic.npy [Content-Type=application/octet-stream]...
- [1 files][  4.3 MiB/  4.3 MiB]  155.2 KiB/s                                   
Operation completed over 1 objects/4.3 MiB.                                      


  Uploaded: X_dynamic.npy


Copying file:///Users/ruben/Desktop/Thesis/2025Data/merged/X_static.npy [Content-Type=application/octet-stream]...
- [1 files][ 87.0 KiB/ 87.0 KiB]                                                
Operation completed over 1 objects/87.0 KiB.                                     


  Uploaded: X_static.npy


Copying file:///Users/ruben/Desktop/Thesis/2025Data/merged/cluster_ids.npy [Content-Type=application/octet-stream]...
/ [1 files][  8.8 KiB/  8.8 KiB]                                                
Operation completed over 1 objects/8.8 KiB.                                      


  Uploaded: cluster_ids.npy


Copying file:///Users/ruben/Desktop/Thesis/2025Data/merged/static_feature_names.txt [Content-Type=text/plain]...


  Uploaded: static_feature_names.txt
All files at: gs://tala-sentinel2-data/data/merged


- [1 files][  367.0 B/  367.0 B]                                                
Operation completed over 1 objects/367.0 B.                                      
